# Lab 12 - Probability Based Learning
## Naive Bayes Classifiers

## 1 Bayes' Theorem

$$P(X|Y) = \frac{P(Y|X) \times P(X)}{P(Y)}$$

Bayes' Theorem defines the conditional probability of an event X given some evidence Y, in terms of
the inverse conditional probability P(Y|X) and the prior probability P(X).


## 2 How Do Naive Bayes Algorithms Work?

Using the weather/play example:

$$P(Yes|Sunny) = \frac{P(Sunny|Yes) \times P(Yes)}{P(Sunny)}$$

With P(Sunny|Yes) = 3/9 = 0.33, P(Sunny) = 5/14 = 0.36, P(Yes) = 9/14 = 0.64:

$$P(Yes|Sunny) = \frac{0.33 \times 0.64}{0.36} = 0.60$$

This is higher than 0.5, so the statement "Players will play if the weather is sunny" is correct.


In [7]:
# Verifying the calculation
P_sunny_given_yes = 3/9
P_yes = 9/14
P_sunny = 5/14

P_yes_given_sunny = (P_sunny_given_yes * P_yes) / P_sunny
print(f"P(Yes|Sunny) = {P_yes_given_sunny:.2f}")

P(Yes|Sunny) = 0.60


## 10 Lab Exercises
### Exercise 1: Apply all types of Naive Bayes on a dataset (using the Iris dataset)

### 1. Gaussian Naive Bayes

In [8]:
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score

iris = load_iris()
X = iris.data
y = iris.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

print('Shape of training data:', X_train.shape)
print('Shape of testing data:', X_test.shape)

model = GaussianNB()
model.fit(X_train, y_train)

predict_train = model.predict(X_train)
accuracy_train = accuracy_score(y_train, predict_train)
print('accuracy_score on train dataset:', accuracy_train)

predict_test = model.predict(X_test)
accuracy_test = accuracy_score(y_test, predict_test)
print('accuracy_score on test dataset:', accuracy_test)

Shape of training data: (105, 4)
Shape of testing data: (45, 4)
accuracy_score on train dataset: 0.9428571428571428
accuracy_score on test dataset: 0.9777777777777777


### 2. Multinomial Naive Bayes (text classification example)

In [9]:
import pandas as pd
import numpy as np

# Create a small sample text classification dataset
data = {
    'message': [
        'I love this sandwich', 'this is an amazing place', 'I feel very good about these beers',
        'this is my best work', 'what an awesome view', 'I do not like this restaurant',
        'I am tired of this stuff', 'I cant deal with this', 'he is my sworn enemy',
        'my boss is horrible', 'this is an awesome place', 'I do not like the taste of this juice',
        'I love to dance', 'I am sick and tired of this place', 'what a great holiday',
        'that is a bad locality to stay', 'we will have good fun tomorrow', 'I went to my enemy house today'
    ],
    'label': ['pos', 'pos', 'pos', 'pos', 'pos', 'neg', 'neg', 'neg', 'neg', 'neg',
              'pos', 'neg', 'pos', 'neg', 'pos', 'neg', 'pos', 'neg']
}

msg = pd.DataFrame(data)
print("Total Instances of Dataset:", msg.shape[0])

msg['labelnum'] = msg.label.map({'pos': 1, 'neg': 0})
X = msg.message
y = msg.labelnum

from sklearn.model_selection import train_test_split
Xtrain, Xtest, ytrain, ytest = train_test_split(X, y, random_state=42)

from sklearn.feature_extraction.text import CountVectorizer
count_v = CountVectorizer()
Xtrain_dm = count_v.fit_transform(Xtrain)
Xtest_dm = count_v.transform(Xtest)

df = pd.DataFrame(Xtrain_dm.toarray(), columns=count_v.get_feature_names_out())
print(df.head())

from sklearn.naive_bayes import MultinomialNB
clf = MultinomialNB()
clf.fit(Xtrain_dm, ytrain)
pred = clf.predict(Xtest_dm)

for doc, p in zip(Xtest, pred):
    p_label = 'pos' if p == 1 else 'neg'
    print("%s -> %s" % (doc, p_label))

from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score
print('\nAccuracy Metrics:\n')
print('Accuracy:', accuracy_score(ytest, pred))
print('Recall:', recall_score(ytest, pred, zero_division=0))
print('Precision:', precision_score(ytest, pred, zero_division=0))
print('Confusion Matrix:\n', confusion_matrix(ytest, pred))

Total Instances of Dataset: 18
   about  am  an  and  awesome  bad  beers  boss  cant  dance  ...  to  today  \
0      0   1   0    1        0    0      0     0     0      0  ...   0      0   
1      0   0   0    0        0    0      0     0     0      0  ...   0      0   
2      0   0   0    0        0    1      0     0     0      0  ...   1      0   
3      0   0   0    0        0    0      0     0     0      0  ...   0      0   
4      1   0   0    0        0    0      1     0     0      0  ...   0      0   

   tomorrow  very  view  we  went  what  will  with  
0         0     0     0   0     0     0     0     0  
1         1     0     0   1     0     0     1     0  
2         0     0     0   0     0     0     0     0  
3         0     0     0   0     0     0     0     0  
4         0     1     0   0     0     0     0     0  

[5 rows x 49 columns]
I love this sandwich -> neg
this is an amazing place -> pos
he is my sworn enemy -> neg
I do not like this restaurant -> neg
this is my

### 3. Bernoulli Naive Bayes

In [10]:
from sklearn.naive_bayes import BernoulliNB
from sklearn.preprocessing import Binarizer

# Re-use the iris dataset, binarizing the features
X_iris_arr = iris.data
y_iris_arr = iris.target

binarizer = Binarizer(threshold=np.mean(X_iris_arr))
X_binarized = binarizer.fit_transform(X_iris_arr)

X_train_b, X_test_b, y_train_b, y_test_b = train_test_split(X_binarized, y_iris_arr, test_size=0.3, random_state=42)

bnb = BernoulliNB()
bnb.fit(X_train_b, y_train_b)

y_pred_bnb = bnb.predict(X_test_b)
print("Bernoulli Naive Bayes model accuracy (in %):", accuracy_score(y_test_b, y_pred_bnb) * 100)

Bernoulli Naive Bayes model accuracy (in %): 71.11111111111111


### 4. Categorical Naive Bayes

In [11]:
from sklearn.naive_bayes import CategoricalNB
from sklearn.preprocessing import KBinsDiscretizer

# Discretize features into categorical bins for CategoricalNB
discretizer = KBinsDiscretizer(n_bins=4, encode='ordinal', strategy='uniform')
X_categorical = discretizer.fit_transform(iris.data)

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X_categorical, iris.target, test_size=0.4, random_state=1)

cnb = CategoricalNB()
cnb.fit(X_train_c, y_train_c)

y_pred_cnb = cnb.predict(X_test_c)
print("Categorical Naive Bayes model accuracy (in %):", accuracy_score(y_test_c, y_pred_cnb) * 100)

Categorical Naive Bayes model accuracy (in %): 93.33333333333333


### 5. Complement Naive Bayes

In [12]:
from sklearn.datasets import load_wine
from sklearn.metrics import classification_report
from sklearn.naive_bayes import ComplementNB

dataset = load_wine()
X_wine = dataset.data
y_wine = dataset.target

X_train_w, X_test_w, y_train_w, y_test_w = train_test_split(X_wine, y_wine, test_size=0.15, random_state=42)

classifier = ComplementNB()
classifier.fit(X_train_w, y_train_w)

prediction = classifier.predict(X_test_w)
prediction_train = classifier.predict(X_train_w)

print(f"Training Set Accuracy: {accuracy_score(y_train_w, prediction_train) * 100}%\n")
print(f"Test Set Accuracy: {accuracy_score(y_test_w, prediction) * 100}%\n")
print(f"Classifier Report:\n\n{classification_report(y_test_w, prediction)}")

Training Set Accuracy: 65.56291390728477%

Test Set Accuracy: 66.66666666666666%

Classifier Report:

              precision    recall  f1-score   support

           0       0.64      1.00      0.78         9
           1       0.67      0.73      0.70        11
           2       1.00      0.14      0.25         7

    accuracy                           0.67        27
   macro avg       0.77      0.62      0.58        27
weighted avg       0.75      0.67      0.61        27



## Summary of Results

| Naive Bayes Variant | Dataset | Notes |
|---|---|---|
| Gaussian NB | Iris (numeric features) | Assumes features follow a normal distribution; well suited for continuous data |
| Multinomial NB | Text classification (word counts) | Suited for discrete count data such as word frequencies |
| Bernoulli NB | Iris (binarized features) | Suited for binary/boolean feature vectors |
| Categorical NB | Iris (discretized into bins) | Suited for categorical features |
| Complement NB | Wine dataset | An adaptation of Multinomial NB suited for imbalanced datasets |

All five Naive Bayes variants were applied successfully. As expected, Gaussian NB performs very
well on the Iris dataset since its features are continuous and roughly normally distributed,
while the other variants required transforming the data (binarizing or discretizing) to match
their assumptions about feature types.
